# Financial Sector

The model of [Brunnermeier and Sannikov (2014)](https://doi.org/10.1257/aer.104.2.379), adapted from an example of EconPDEs.jl. Experts and households both hold capital, but experts are more productive and can issue equity only if they retain at least a fraction `chi` of it. The state `x` is the experts' wealth share.

There is a crisis region, with experts constrained and `Psi < 1`, and a normal region, with `Psi = 1`, where `Psi` is the share of aggregate capital operated by experts.

In [ ]:
Pollis.packages("EconPDEs", "Roots", "Plots")

## From EconPDEs.jl

The code is the package's example, with Greek letters written in Latin (`rho`, `gamma`, `psi`, `delta`, `sigma`, `kappa`, `chi`, `Psi`).

### The model

The capital price `q` and `Psi` are found by a bracketed root-find, sweeping `x` upward from the lower boundary, so that each step reuses the previous `q` to build its derivative. That is why the struct is mutable and carries `q_old`.

In [ ]:
using EconPDEs, Roots, Plots

Base.@kwdef mutable struct BrunnermeierSannikov
  # Calibration uses Section 3.6 of the textbook chapter (the paper's r here appears to be a typo for rhoH)
  rhoE::Float64 = 0.06      # experts' rate of time preference (discount rate)
  rhoH::Float64 = 0.05      # households' rate of time preference (discount rate)
  gamma::Float64 = 2.0        # relative risk aversion
  psi::Float64 = 0.5        # elasticity of intertemporal substitution
  delta::Float64 = 0.05       # depreciation rate of capital
  sigma::Float64 = 0.1        # calibrated fundamental volatility of capital
  kappa::Float64 = 10.0       # investment adjustment-cost parameter
  a::Float64 = 0.11       # experts' productivity (output per unit capital)
  alow::Float64 = 0.03    # households' productivity (output per unit capital)
  chilow::Float64 = 0.5     # minimum equity retention (skin-in-the-game floor)
  q_old::Float64 = 1.0    # previous capital price q (carried across the sweep)
  qx_old::Float64 = 0.0   # previous q_x (carried across the sweep)
  Psi_old::Float64 = 1.0    # previous experts' capital share Psi (carried across the sweep)
end

In [ ]:
m = BrunnermeierSannikov()

### The grid and the guess

The state has 200 interior points in (0, 1). The grid spacing and its minimum are passed to the model later, for the sweep. The two unknowns are `pE` and `pH`, the wealth-consumption ratios of experts and households.

In [ ]:
xn = 200
stategrid =  (; x = range(0, 1.0, length = xn+2)[2:(end-1)])
Deltax = step(stategrid[:x])
xmin = minimum(stategrid[:x])

In [ ]:
guess = (; pE =  9 .* ones(xn), pH =   10 .* ones(xn))

### The equations

A small helper finds a sign change for the root-find of the crisis region, starting from the previous `Psi`. Then the function recovers `q` and `Psi`, separating the two regions, and returns the time derivatives `pEt` and `pHt`.

In [ ]:
function local_bracket(f, x0; lo = 0.0, hi = 1.0, ftol = 1e-12)
  x0 = clamp(x0, lo, hi)
  f0 = f(x0)
  isfinite(f0) || return nothing
  abs(f0) <= ftol && return (x0, x0)
  width = (hi - lo) / 64
  while width <= hi - lo
    for x1 in (min(hi, x0 + width), max(lo, x0 - width))
      f1 = f(x1)
      isfinite(f1) || continue
      abs(f1) <= ftol && return (x1, x1)
      signbit(f1) != signbit(f0) && return minmax(x0, x1)
    end
    width *= 2
  end
  return nothing
end

In [ ]:
function (m::BrunnermeierSannikov)(state::NamedTuple, u::NamedTuple, Deltax, xmin)
  (; rhoE, rhoH, gamma, psi, delta, sigma, kappa, a, alow, chilow) = m
  (; x) = state
  (; pE, pEx_up, pEx_down, pExx, pH, pHx_up, pHx_down, pHxx) = u
  # Floor the wealth-consumption ratios before dividing by them: a Newton trial iterate can wander
  # nonpositive, which would blow up the equilibrium conditions. Slack at the converged solution,
  # where both stay well above 1e-3.
  pE = max(pE, 1e-3)
  pH = max(pH, 1e-3)
  chi = max(chilow, x)
  pEx, pHx = pEx_up, pHx_up
  iter = 0
  @label start
  q_old = m.q_old
  qx_old = m.qx_old
  Psi_old = m.Psi_old

  # First, solve for q, qx, and qxx
  if abs(x - xmin) < 1e-9
    # initial condition at lower boundary of x grid (point outside the grid technically)
    # We must have that expert consumption approaches zero as xt approaches zero (linked to transversality condition)
    Psi_old = 0.0
    q_old = (alow + 1 / kappa) / (1 / pH + 1 / kappa)
    qx_old = 0.0
  end
  if Psi_old < 1.0
    # crisis region.
    # In this case experts operate only the share Psi of aggregate capital. They retain chiPsi of
    # aggregate capital, so their capital holding per unit of expert wealth is alphaE = chiPsi / x.
    function crisis_gap(Psi)
      local q = (Psi * a + (1 - Psi) * alow + 1 / kappa) / (x / pE + (1 - x) / pH + 1 / kappa)
      local qx = (q - q_old) / Deltax
      local alphaE = chi * Psi / x
      local sigmax = x * (alphaE - 1) * sigma / (1 - x * (alphaE - 1) * qx / q)
      local sigmapE = pEx / pE * sigmax
      local sigmapH = pHx / pH * sigmax
      local sigmaq = qx / q * sigmax
      local sigmaE = alphaE * (sigma + sigmaq)
      local alphaH = (1 - alphaE * x) / (1 - x)
      local sigmaH = alphaH * (sigma + sigmaq)
      local kappaE = gamma * (sigmaE - (1 / gamma - 1) / (psi - 1) * sigmapE)
      local kappaH = gamma * (sigmaH - (1 / gamma - 1) / (psi - 1) * sigmapH)
      return (a - alow) / q - chi * (kappaE - kappaH) * (sigma + sigmaq)
    end

    gap_tolerance = 1e-12
    bracket = local_bracket(crisis_gap, Psi_old; ftol = gap_tolerance)

    if bracket === nothing
      # The unconstrained crisis equation has no root with Psi in [0, 1], so the economy is in
      # the normal region and experts operate all capital.
      Psi_old = 1.0
    else
      Psi = bracket[1]
      if bracket[1] != bracket[2]
        Psi = find_zero(crisis_gap, bracket, Bisection(); xatol = gap_tolerance)
      end
      q = (Psi * a + (1 - Psi) * alow + 1 / kappa) / (x / pE + (1 - x) / pH + 1 / kappa)
      qx = (q - q_old) / Deltax
    end
  end
  if Psi_old >= 1.0
    # normal region.
    Psi = 1.0
    q = (a + 1 / kappa) / ((x / pE + (1 - x) / pH) + 1 / kappa)
    qx = - (a + 1 / kappa) / ((x / pE + (1 - x) / pH) + 1 / kappa)^2 * (1 / pE - 1 / pH - x / pE^2 * pEx - (1-x) / pH^2 * pHx)
  end
  qxx = (qx - qx_old) / Deltax
  m.q_old = q
  m.qx_old = qx
  m.Psi_old = Psi


  # having solved for q, do the time step
  # Floor q the same way before it enters iota, Phi, muq, and r; slack at the converged solution (q about 1).
  q = max(q, sqrt(eps()))
  iota = (q - 1) / kappa
  Phi = log(q) / kappa
  alphaE = chi * Psi / x
  sigmax = x * (alphaE - 1) * sigma / (1 - x * (alphaE - 1) * qx / q)
  sigmapE = pEx / pE * sigmax
  sigmapH = pHx / pH * sigmax
  sigmaq = qx / q * sigmax
  sigmaE = alphaE * (sigma + sigmaq)
  alphaH = (1 - alphaE * x) / (1 - x)
  sigmaH = alphaH * (sigma + sigmaq)
  kappaE = gamma * (sigmaE - (1 / gamma - 1) / (psi - 1) * sigmapE)
  kappaH = gamma * (sigmaH - (1 / gamma - 1) / (psi - 1) * sigmapH)
  mux = x * (1 - x) * (kappaE * sigmaE - kappaH * sigmaH + 1 / pH - 1 / pE - (sigmaE - sigmaH) * (sigma + sigmaq))
  mux2 = x * ((a - iota) / q - 1 / pE) + sigmax * (kappaE - sigma - sigmaq) + x * (1 - chi) * (kappaE - kappaH) * (sigma + sigmaq)
  if (iter == 0) && (mux < 0)
    # upwinding
    iter += 1
    pEx, pHx = pEx_down, pHx_down
    @goto start
  end
  mupE = pEx / pE * mux + 0.5 * pExx / pE * sigmax^2
  mupH = pHx / pH * mux + 0.5 * pHxx / pH * sigmax^2
  muq = qx / q * mux + 0.5 * qxx / q * sigmax^2

  # Clamp r to a wide band: a Newton trial iterate outside the economic domain can produce a wild
  # interest rate that would overflow the exponential price updates below. Slack at the converged
  # solution, where r stays within a few percent of zero.
  r = clamp((a - iota) / q + Phi - delta + muq + sigma * sigmaq - (chi * kappaE + (1 - chi) * kappaH) * (sigma + sigmaq), -3.0, 3.0)
  # Market Pricing
  pEt = -  pE * (1 / pE - psi * rhoE + (psi - 1) * (r + kappaE * sigmaE) + mupE - (psi - 1) * gamma / 2 * sigmaE^2 + (2 - psi - gamma) / (2 * (psi - 1)) * sigmapE^2 + (1 - gamma) * sigmapE * sigmaE)
  pHt = -  pH * (1 / pH - psi * rhoH + (psi - 1) * (r + kappaH * sigmaH) + mupH - (psi - 1) * gamma / 2 * sigmaH^2 + (2 - psi - gamma) / (2 * (psi - 1)) * sigmapH^2 + (1 - gamma) * sigmapH * sigmaH)
  d = Psi * a + (1 - Psi) * alow - iota
  muR = r + (chi * kappaE + (1 - chi) * kappaH) * (sigma + sigmaq)
  (; pEt, pHt), (; x, r, mux, sigmax, Psi, kappa, kappaE, kappaH, sigmaE, sigmaH, sigmaq, q, qx, qxx, muq, mux2, Phi, chi, d, muR)
end

### Solving it

The grid spacing and the lower bound reach the model through a closure.

In [ ]:
result = pdesolve((state, u) -> m(state, u, Deltax, xmin), stategrid, guess)

### The solution

The capital price is depressed when experts are poorly capitalized, in the crisis region, and rises with their wealth share as the retention constraint slackens.

In [ ]:
xs = stategrid[:x]
plot(xs, result.saved.q; xlabel = "experts' wealth share x", ylabel = "capital price q", legend = false)

## Source and Licence

The code under **From EconPDEs.jl** is adapted from `examples/asset_pricing/brunnermeier_sannikov.jl` of [EconPDEs.jl](https://github.com/matthieugomez/EconPDEs.jl), version 3.1.0, by Matthieu Gomez. Changes made in Pollis: Greek letters written in Latin.

The notice below covers the package's code only, not the parts added in Pollis.

The EconPDEs.jl package is licensed under the MIT "Expat" License:

> Copyright (c) 2015: Matthieu Gomez.
>
> Permission is hereby granted, free of charge, to any person obtaining
> a copy of this software and associated documentation files (the
> "Software"), to deal in the Software without restriction, including
> without limitation the rights to use, copy, modify, merge, publish,
> distribute, sublicense, and/or sell copies of the Software, and to
> permit persons to whom the Software is furnished to do so, subject to
> the following conditions:
>
> The above copyright notice and this permission notice shall be
> included in all copies or substantial portions of the Software.
>
> THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
> EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
> MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
> IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
> CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
> TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
> SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## References

Brunnermeier, M. K. and Sannikov, Y. (2014). A macroeconomic model with a financial sector. *American Economic Review* 104(2), 379–421. [doi:10.1257/aer.104.2.379](https://doi.org/10.1257/aer.104.2.379)